# Treasure Hunt Game - CS 499 Algorithms and Data Structures Enhancement

**Savanna Neace**

This enhanced artifact keeps the original maze and deep Q-learning design and adds an A* pathfinding implementation. A* provides a deterministic search-based solution that can be compared with the learned policy from the original project.

## Enhancement goals

- Add A* search using a priority queue, sets, and dictionaries.
- Use Manhattan distance as the heuristic.
- Report path length/cost and number of explored states.
- Visualize the A* route through the same maze.
- Add a helper that can compare A* with the trained deep Q-learning agent.
- Repair the missing training helper functions from the exported original notebook so the deep Q-learning portion is complete.

In [ ]:
import os, time, datetime, random
import numpy as np
import matplotlib.pyplot as plt

from TreasureMaze_Enhanced import TreasureMaze
from AStarPathfinding import astar_search

LEFT, UP, RIGHT, DOWN = 0, 1, 2, 3
actions_dict = {LEFT: 'left', UP: 'up', RIGHT: 'right', DOWN: 'down'}
num_actions = len(actions_dict)

In [ ]:
maze = np.array([
    [1., 0., 1., 1., 1., 1., 1., 1.],
    [1., 0., 1., 1., 1., 0., 1., 1.],
    [1., 1., 1., 1., 0., 1., 0., 1.],
    [1., 1., 1., 0., 1., 1., 1., 1.],
    [1., 1., 0., 1., 1., 1., 1., 1.],
    [1., 1., 1., 0., 1., 0., 0., 0.],
    [1., 1., 1., 0., 1., 1., 1., 1.],
    [1., 1., 1., 1., 0., 1., 1., 1.]
])
qmaze = TreasureMaze(maze)
print('Treasure location:', qmaze.target)

## A* result

For the original start cell `(0, 0)`, the A* enhancement finds a valid route to the treasure at `(7, 7)` in **24 moves**. In the tested maze, it explored **50 states**. This gives the project a clear, reproducible baseline for evaluating the learned agent.

In [1]:
start = (0, 0)
path, stats = qmaze.astar_path(start=start)

print('A* path:', path)
print('Moves:', stats['moves'])
print('Path cost:', stats['path_cost'])
print('States explored:', stats['states_explored'])

A* path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (1, 3), (1, 4), (0, 4), (0, 5), (0, 6), (0, 7), (1, 7), (2, 7), (3, 7), (4, 7), (4, 6), (4, 5), (4, 4), (5, 4), (6, 4), (6, 5), (6, 6), (6, 7), (7, 7)]
Moves: 24
Path cost: 24
States explored: 50


In [ ]:
def show_astar_path(maze, path):
    canvas = np.array(maze, copy=True)
    if path:
        for row, col in path[1:-1]:
            canvas[row, col] = 0.55
        sr, sc = path[0]
        gr, gc = path[-1]
        canvas[sr, sc] = 0.30
        canvas[gr, gc] = 0.90

    plt.figure(figsize=(6, 6))
    plt.imshow(canvas, interpolation='none', cmap='gray', vmin=0, vmax=1)
    plt.title('A* Path Through the Treasure Maze')
    plt.xticks(range(maze.shape[1]))
    plt.yticks(range(maze.shape[0]))
    plt.grid(True)
    plt.show()

show_astar_path(maze, path)

## Deep Q-learning compatibility and comparison

The original artifact uses deep Q-learning. The following helpers repair two missing functions from the exported notebook (`format_time` and the model training step) and provide a path-capture function. After the model is trained, `compare_algorithms` reports the A* and learned-agent path metrics side by side.

In [ ]:
# These imports are only needed for the original deep Q-learning portion.
import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, PReLU, Input
from GameExperience_Enhanced import GameExperience

loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

def format_time(seconds):
    seconds = int(seconds)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def train_step(model, x, y):
    with tf.GradientTape() as tape:
        predictions = model(x, training=True)
        loss_value = loss_fn(y, predictions)
    gradients = tape.gradient(loss_value, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return float(loss_value.numpy())

def build_model(maze):
    model = Sequential([
        Input(shape=(maze.size,)),
        Dense(maze.size),
        PReLU(),
        Dense(maze.size),
        PReLU(),
        Dense(num_actions),
    ])
    return model

def play_game_with_path(model, maze, start=(0, 0), max_steps=None):
    qmaze = TreasureMaze(maze, pirate=start)
    envstate = qmaze.observe()
    path = [start]
    if max_steps is None:
        max_steps = maze.size * 4

    for _ in range(max_steps):
        state = np.asarray(envstate, dtype=np.float32)
        if state.ndim == 1:
            state = np.expand_dims(state, axis=0)
        q_values = model(state, training=False).numpy()[0]
        valid_actions = qmaze.valid_actions()
        masked = np.full(num_actions, -np.inf)
        for action in valid_actions:
            masked[action] = q_values[action]
        action = int(np.argmax(masked))
        envstate, reward, status = qmaze.act(action)
        path.append((qmaze.state[0], qmaze.state[1]))
        if status == 'win':
            return path, True
        if status == 'lose':
            return path, False
    return path, False

def compare_algorithms(model, maze, start=(0, 0)):
    a_path, a_stats = astar_search(maze, start=start)
    q_path, q_won = play_game_with_path(model, maze, start=start)
    return {
        'A*': {
            'reached_treasure': a_path is not None,
            'moves': a_stats['moves'],
            'states_explored': a_stats['states_explored'],
        },
        'Deep Q-learning': {
            'reached_treasure': q_won,
            'moves': len(q_path) - 1,
            'states_explored': 'learned policy (not directly comparable)',
        },
    }


In [ ]:
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

def completion_check(model, maze_or_qmaze, max_steps=None):
    qmaze = maze_or_qmaze if isinstance(maze_or_qmaze, TreasureMaze) else TreasureMaze(maze_or_qmaze)
    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            continue
        _, won = play_game_with_path(model, qmaze._maze, start=cell, max_steps=max_steps)
        if not won:
            return False
    return True

def qtrain(model, maze, **opt):
    global epsilon
    n_epoch = opt.get('n_epoch', 1000)
    max_memory = opt.get('max_memory', 8 * maze.size)
    data_size = opt.get('data_size', 32)
    target_update_freq = opt.get('target_update_freq', 50)

    start_time = datetime.datetime.now()
    qmaze = TreasureMaze(maze)
    target_model = clone_model(model)
    target_model.set_weights(model.get_weights())
    experience = GameExperience(model, target_model, max_memory=max_memory)

    win_history = []
    hsize = qmaze.maze.size // 2
    loss = 0.0

    for epoch in range(n_epoch):
        pirate_cell = random.choice(qmaze.free_cells)
        qmaze.reset(pirate_cell)
        envstate = qmaze.observe()
        status = 'not_over'

        while status == 'not_over':
            previous = envstate
            valid_actions = qmaze.valid_actions()
            if np.random.rand() < epsilon:
                action = random.choice(valid_actions)
            else:
                q_values = model(np.asarray(previous, dtype=np.float32), training=False).numpy()[0]
                masked = np.full(num_actions, -np.inf)
                for a in valid_actions:
                    masked[a] = q_values[a]
                action = int(np.argmax(masked))

            envstate, reward, status = qmaze.act(action)
            if status == 'win':
                win_history.append(1)
            elif status == 'lose':
                win_history.append(0)

            experience.remember([previous, action, reward, envstate, status])
            inputs, targets = experience.get_data(data_size)
            loss = train_step(
                model,
                tf.convert_to_tensor(inputs, dtype=tf.float32),
                tf.convert_to_tensor(targets, dtype=tf.float32),
            )

        if epoch % target_update_freq == 0:
            target_model.set_weights(model.get_weights())

        win_rate = sum(win_history[-hsize:]) / hsize if len(win_history) >= hsize else 0.0
        epsilon = 0.05 if win_rate > 0.9 else max(epsilon * epsilon_decay, epsilon_min)

        if epoch % 10 == 0 or win_rate >= 0.999:
            elapsed = format_time((datetime.datetime.now() - start_time).total_seconds())
            print(f'Epoch {epoch:04d} | loss {loss:.4f} | win rate {win_rate:.3f} | {elapsed}')

        if win_rate >= 0.999 and completion_check(model, maze):
            print(f'Completion check passed at epoch {epoch}.')
            break

    return model


### Optional training/comparison

Run the next cell when TensorFlow is available. Training time depends on the computer. The A* enhancement itself does not require neural-network training.

In [ ]:
# model = build_model(maze)
# model = qtrain(model, maze, n_epoch=1000)
# compare_algorithms(model, maze, start=(0, 0))

## Reflection on the enhancement

A* and deep Q-learning solve the maze in different ways. A* searches the maze directly and uses the Manhattan-distance heuristic to prioritize promising cells. The learned agent instead chooses actions from values developed during training. Adding A* makes the artifact stronger because it demonstrates algorithm selection, priority queues, sets, dictionaries, heuristics, path reconstruction, and measurable trade-offs between a deterministic search and a learned policy.